In [1]:
import os
import geopandas as gpd
import pandas as pd
import datetime
from pathlib import Path
import subprocess

In [2]:
# CELL 2 – Temporary workaround for r5py/Python 3.12 compatibility issue with Java/JPype
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.11'

# Patch jpype before importing to handle bytes return from getDefaultJVMPath
import jpype
original_getDefaultJVMPath = jpype.getDefaultJVMPath

def patched_getDefaultJVMPath(*args, **kwargs):
    result = original_getDefaultJVMPath(*args, **kwargs)
    if isinstance(result, bytes):
        result = result.decode('utf-8')
    return result

jpype.getDefaultJVMPath = patched_getDefaultJVMPath

# Import r5py after patching jpype
import r5py

In [3]:
# CELL 3 – Load data

OSM_FILE = "../../data/geo/Toronto.osm.pbf"
GTFS_FILES = [
    "../../data/geo/TTC Routes and Schedules Data.zip",
    "../../data/geo/GO-GTFS.zip",
    "../../data/geo/UP-GTFS.zip"
]
ORIGINS_FILE = "../../src/data/venues-centroids.geo.json"
LAKE_ONTARIO_FILE = "../../data/geo/lake-ontario.gpkg"

In [4]:
# CELL 4 - Set parameters

# Isochrone cutoffs (minutes)
isochrone_cutoffs = [15, 30, 45, 60]

# Resolution (meters)
point_grid_resolution = 150

# Departure time
departure_time = datetime.datetime(
    2026, 6, 9,
    14, 0, 0
    )

In [5]:
# CELL 5 – Load origins
origins = gpd.read_file(ORIGINS_FILE)
origins["id"] = origins["id"].astype(str)
origins = origins.to_crs("EPSG:4326")

In [6]:
# CELL 6 – Build transport network
transport_network = r5py.TransportNetwork(
    OSM_FILE,
    GTFS_FILES
)
transport_network

In [7]:
# CELL 7 – Compute isochrones for each origin

isochrone_frames = []

for _, origin in origins.iterrows():
    origin_id = origin["id"]
    origin_gdf = gpd.GeoDataFrame(
        [{"id": origin_id}],
        geometry=[origin.geometry],
        crs="EPSG:4326"
    )

    isochrones = r5py.Isochrones(
        transport_network,
        origins=origin_gdf,
        departure=departure_time,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        isochrones=isochrone_cutoffs,
        point_grid_resolution=point_grid_resolution,
    )

    isochrones["origin_id"] = origin_id
    # Convert the Timedelta travel time into plain minutes for easier
    # styling/filtering downstream (e.g. in MapLibre/Tippecanoe)
    isochrones["travel_time_minutes"] = (
        isochrones["travel_time"].dt.total_seconds() / 60
    )

    isochrone_frames.append(isochrones)

    print(f"Done: {origin_id}")

isochrones_all = gpd.GeoDataFrame(
    pd.concat(isochrone_frames, ignore_index=True),
    crs=isochrone_frames[0].crs
).drop(columns="travel_time")

Done: 1
Done: 2
Done: 3
Done: 4
Done: 5
Done: 6
Done: 7
Done: 8
Done: 9
Done: 10
Done: 11
Done: 12
Done: 13
Done: 14
Done: 15
Done: 16
Done: 17
Done: 18
Done: 19
Done: 20
Done: 21
Done: 22
Done: 23
Done: 24
Done: 25
Done: 26
Done: 27
Done: 28
Done: 29
Done: 30
Done: 31
Done: 32
Done: 33
Done: 34
Done: 35
Done: 36
Done: 37
Done: 38
Done: 39
Done: 40
Done: 41
Done: 42
Done: 43
Done: 44
Done: 45
Done: 46
Done: 47
Done: 48
Done: 49
Done: 50
Done: 51
Done: 52
Done: 53
Done: 54
Done: 55
Done: 56
Done: 57
Done: 58
Done: 59
Done: 60
Done: 61
Done: 62
Done: 63
Done: 64
Done: 65
Done: 66
Done: 67
Done: 68
Done: 69
Done: 70
Done: 71
Done: 72
Done: 73
Done: 74
Done: 75
Done: 76
Done: 77
Done: 78
Done: 79
Done: 80
Done: 81
Done: 82
Done: 83


In [8]:
# CELL 8 – (optional) clip isochrone lines so they don't extend out over Lake Ontario
if Path(LAKE_ONTARIO_FILE).exists():
    lake_ontario = gpd.read_file(LAKE_ONTARIO_FILE).to_crs(isochrones_all.crs)
    isochrones_all["geometry"] = isochrones_all.geometry.difference(
        lake_ontario.geometry.union_all()
    )
    # Drop any rows that became empty after clipping
    isochrones_all = isochrones_all[~isochrones_all.geometry.is_empty]

In [9]:
# CELL 9 – Export isochrones as a tileset

OUTPUT_FILE = "../../data/geo/commute_isochrones.geojson"
isochrones_all.to_file(OUTPUT_FILE, driver="geojson")

# PMTILES_FILE = "../../static/commute_isochrones.pmtiles"

# subprocess.run([
#     "tippecanoe",
#     "-z17",
#     "-o", PMTILES_FILE,
#     "--no-feature-limit",
#     "--no-tile-size-limit",
#     OUTPUT_FILE,
#     "--force"
# ], check=True)

# Path(OUTPUT_FILE).unlink()